# 2026-11-08 Scaling: exact 1-NN search time vs database size

**Fixed:** 3 query sets (1000 XFeat descriptors each, COCO test images), database = HPatches XFeat descriptors (first N), exact 1-NN (smallest index on ties), 2 threads, database encoded offline (not timed), cache-blocked loops for every method.

**Changed:** database size N ∈ {1k, 4k, 16k, 64k, 256k, 512k} and representation/distance: `float32_blas`, `int8_dot`, `byte8_L1`, `byte8_L2`, `nib16_L2_packed` (4-bit, 32 B), `thermo8_L1` (56 B).

Speed only: no accuracy, no setting is selected here.

**Before running:** upload this folder to `MyDrive/Colab Notebooks/2026-11-08_scaling/`. The cache `MyDrive/Colab Notebooks/2026-10-14_regen_cache/` is already there.
Runtime: CPU. The cloud test took about 2 min; Colab may take longer (not measured).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
BASE=Path('/content/drive/MyDrive/Colab Notebooks')
EXP=BASE/'2026-11-08_scaling'; CACHE=BASE/'2026-10-14_regen_cache'
for p in [EXP/'run_scaling.py',EXP/'scale_kernels.py',EXP/'inputs'/'edges_all.npz',EXP/'inputs'/'fixed_quantile_edges.npz',
          CACHE/'coco'/'test_pairs.npz',CACHE/'hpatches_cache'/'hpatches_pairs.npz']:
    assert p.exists(),p
print('inputs found')

In [ ]:
import subprocess,sys,os
subprocess.run(['rm','-rf','/content/run','/content/cache'],check=True)
subprocess.run(['cp','-r',str(EXP),'/content/run'],check=True)
subprocess.run(['cp','-r',str(CACHE),'/content/cache'],check=True)
print(subprocess.run("lscpu | grep -E 'Model name|^CPU\\(s\\)|cache'",shell=True,capture_output=True,text=True).stdout)
print('avx2' in open('/proc/cpuinfo').read())

In [ ]:
import time
env=dict(os.environ,OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',MKL_NUM_THREADS='2')
start=time.time()
p=subprocess.Popen([sys.executable,'-u','/content/run/run_scaling.py','/content/cache','/content/run/inputs','/content/run/results_colab'],
                   stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,env=env)
for line in p.stdout: print(line,end='')   # live log
p.wait(); print('minutes',round((time.time()-start)/60,1)); assert p.returncode==0

In [ ]:
out=EXP/f'results_colab_{time.strftime("%Y%m%d_%H%M%S")}'
subprocess.run(['cp','-r','/content/run/results_colab',str(out)],check=True)
print('saved to',out)